# Datasets download pipeline documentation

This notebook documents the dataset download pipeline used in the Audio MIDI project.

It explains:
- the pipeline architecture
- the execution flow
- the available CLI options
- how to run the pipeline
- the download and extraction workflow
- idempotency behavior

## Pipeline Responsibilities

`src.pipelines.DatasetsDownloadPipeline` is responsible for:

1. Downloading dataset archives
2. Extracting ZIP archives
3. Collecting execution statistics

The pipeline currently supports:

- GuitarSet
- IDMT-SMT-Guitar

## Pipeline Construction

The pipeline is initialized from `./audio_midi/main.py`.

```python
download_datasets_pipeline = DatasetsDownloadPipeline(
    logger=logger,
    guitar_set=args.guitar_set,
    idmt_smt_guitar=args.idmt_smt_guitar,
)
```

### Parameters

| Parameter | Description |
| :- | :- |
| `logger` | Shared application logger |
| `guitar_set` | Enable GuitarSet download |
| `idmt_smt_guitar` | Enable IDMT-SMT-Guitar download |


## Execution Flow

The pipeline execution flow is:

```text
main.py
 └── DatasetsDownloadPipeline.run()
      ├── _process_dataset(config)
      │    ├── _download(...)
      │    └── _extract(...)
      └── statistics reporting
```


## Download Workflow

The download step uses `src.downloaders.DatasetDownloader`.

Features:

- configurable HTTP retry strategy
- resumable downloads using HTTP Range headers
- temporary .part files for partial downloads
- automatic retry with exponential backoff
- configurable HTTP session and headers
- progress tracking with tqdm

Downloads are skipped when the target archive already exists.

If a partial .part file exists, the downloader attempts to resume the transfer instead of restarting from scratch.


## Extraction Workflow

The extraction step uses `src.extractors.ZipExtractor`.

Features:

- safe extraction into a temporary directory
- Zip Slip protection
- idempotent extraction
- automatic cleanup of temporary directories

Extraction is skipped when the target directory already exists.


## Running the Pipeline from CLI

### Download all datasets

```bash
uv run ./audio_midi/main.py --download_datasets
```

### Download only GuitarSet

```bash
uv run ./audio_midi/main.py --download_datasets --no_idmt_smt_guitar
```

### Download only IDMT-SMT-Guitar

```bash
uv run ./audio_midi/main.py --download_datasets --no_guitar_set
```


## Relevant CLI Arguments

| Argument | Description |
|---|---|
| `--download_datasets` | Run dataset download pipeline |
| `--no_guitar_set` | Disable GuitarSet download |
| `--no_idmt_smt_guitar` | Disable IDMT-SMT-Guitar download |


## Expected Directory Structure

Downloaded archives and extracted datasets are stored under:

```text
./audio_midi/data/
```

Typical structure:

```text
audio_midi/
└── data/
    ├── archive.zip
    └── extracted_dataset/
```


## Error Handling

The pipeline logs and propagates failures occurring during:

- dataset download
- archive extraction
- filesystem operations

Statistics counters are updated accordingly.


## Statistics

The pipeline tracks:

- successful downloads
- failed downloads
- successful extractions
- failed extractions

Statistics are reported at the end of the execution.
